In [0]:
tables = ["outlets", "menu_items", "staff", "customers", "currency_conversion_rates", "orders", "order_items"]

for table_name in tables:
    volume_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/full_load/{table_name}"
    checkpoint_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/_checkpoints/full_load_{table_name}"
    schema_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/_schemas/full_load_{table_name}"
    bronze_table = f"zaitoon_catalog.bronze.{table_name}"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("multiLine", "true")
        .load(volume_path)
    )

    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
    )

    print(f"Loaded {table_name} into {bronze_table}")

In [0]:
for table_name in tables:
    count = spark.table(f"zaitoon_catalog.bronze.{table_name}").count()
    print(f"{table_name}: {count} rows")